# 🚀 Antigravity AI App Builder

This notebook sets up and runs the complete Antigravity system on Google Colab:
- Clones the GitHub repository
- Installs Node.js, builds the React frontend
- Installs Python dependencies
- Compiles the C++ Builder Engine
- Loads GLM-5 onto the GPU
- Starts FastAPI and exposes it via Cloudflare Tunnel

### ⚠️ Prerequisites
1. Set **Runtime → Change runtime type → T4 GPU**
2. Update `REPO_URL` below to point to your GitHub fork

In [ ]:
# ── Configuration ──────────────────────────
REPO_URL = 'https://github.com/YOUR_USERNAME/YOUR_REPO.git'
BRANCH   = 'main'

In [ ]:
# ── Step 1: Clone Repository ───────────────
!rm -rf app_builder
!git clone --branch $BRANCH --depth 1 $REPO_URL app_builder
%cd app_builder
!ls -la

In [ ]:
# ── Step 2: Install Node.js via NVM ────────
!curl -o- https://raw.githubusercontent.com/nvm-sh/nvm/v0.39.5/install.sh | bash
import os
os.environ['NVM_DIR'] = os.path.expanduser('~/.nvm')
!bash -c 'source $NVM_DIR/nvm.sh && nvm install 18 && nvm use 18 && node -v && npm -v'

In [ ]:
# ── Step 3: Build React Frontend ───────────
!bash -c 'source $NVM_DIR/nvm.sh && nvm use 18 && cd frontend && npm install && npm install react-router-dom lucide-react && npm run build'
!ls -la frontend/dist/

In [ ]:
# ── Step 4: Install Python Dependencies ────
!pip install -q fastapi uvicorn transformers torch accelerate

In [ ]:
# ── Step 5: Compile C++ Builder Engine ─────
%cd engine
!curl -L -o json.hpp https://github.com/nlohmann/json/releases/download/v3.11.2/json.hpp
!g++ -std=c++17 -O3 -o builder builder.cpp
!ls -la builder
%cd ..

In [ ]:
# ── Step 6: Verify GPU ─────────────────────
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

In [ ]:
# ── Step 7: Start Backend + Tunnel ─────────
# This will:
#   1. Start FastAPI (loads GLM-5 into GPU)
#   2. Download cloudflared and open a public tunnel
#   3. Print the public URL
!python start.py